# Data Coverage Analysis for Turkish Legal RAG

This notebook analyzes whether the QA datasets and the retrieval corpus cover the same legal sources.

Main goals:
- Inspect train/validation/test QA files.
- Inspect retrieval corpus source coverage.
- Detect source mismatch between QA datasets and retrieval corpus.
- Analyze KVKK-related questions.
- Analyze whether problematic evaluation samples are answerable from the current corpus.
- Produce coverage-aware evaluation notes for later experiments.

In [13]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
project_path = "/content/drive/MyDrive/turkish_legal_rag"

raw_path = f"{project_path}/data/raw"
processed_path = f"{project_path}/data/processed"
metrics_path = f"{project_path}/outputs/metrics"

print("Project path:", project_path)
print("Raw path:", raw_path)
print("Processed path:", processed_path)
print("Metrics path:", metrics_path)

Project path: /content/drive/MyDrive/turkish_legal_rag
Raw path: /content/drive/MyDrive/turkish_legal_rag/data/raw
Processed path: /content/drive/MyDrive/turkish_legal_rag/data/processed
Metrics path: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics


In [15]:
import os
import re
import glob

import numpy as np
import pandas as pd

In [16]:
kaggle_train_df = pd.read_csv(f"{processed_path}/kaggle_train_qa.csv")
kaggle_val_df = pd.read_csv(f"{processed_path}/kaggle_val_qa.csv")
kaggle_test_df = pd.read_csv(f"{processed_path}/kaggle_test_qa.csv")

train_df = pd.read_csv(f"{processed_path}/train_qa.csv")
val_df = pd.read_csv(f"{processed_path}/val_qa.csv")
test_df = pd.read_csv(f"{processed_path}/test_qa.csv")

corpus_df = pd.read_csv(f"{processed_path}/retrieval_corpus.csv")

print("kaggle_train:", kaggle_train_df.shape)
print("kaggle_val:", kaggle_val_df.shape)
print("kaggle_test:", kaggle_test_df.shape)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

print("retrieval corpus:", corpus_df.shape)

kaggle_train: (9019, 6)
kaggle_val: (1933, 6)
kaggle_test: (1933, 6)
train: (11350, 2)
val: (2004, 2)
test: (1500, 2)
retrieval corpus: (3775, 5)


In [17]:
files_to_inspect = {
    "kaggle_train_qa.csv": kaggle_train_df,
    "kaggle_val_qa.csv": kaggle_val_df,
    "kaggle_test_qa.csv": kaggle_test_df,
    "train_qa.csv": train_df,
    "val_qa.csv": val_df,
    "test_qa.csv": test_df,
    "retrieval_corpus.csv": corpus_df
}

for name, df in files_to_inspect.items():
    print("=" * 100)
    print(name)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(2))

kaggle_train_qa.csv
Shape: (9019, 6)
Columns: ['soru', 'cevap', 'veri türü', 'kaynak', 'context', 'score']


,soru,cevap,veri türü,kaynak,context,score
0,"Kanunda uygulanabilir bir hüküm yoksa, hâkim n...","Kanunda uygulanabilir bir hüküm yoksa, hâkim ö...",hukuk,Türk Medeni Kanunu,TÜRK MEDENİ KANUNU\r\nKanun Numarası : 4721\r\...,8
1,"Yargıtay Genel Kurulu, Yargıtay üyeleri arasın...","Yargıtay Genel Kurulu, Yargıtay üyeleri arasın...",hukuk,Türkiye Cumhuriyeti Anayasası,ALTINCI KISIM\n\nGEÇİCİ HÜKÜMLER\nGeçici Madde...,10


kaggle_val_qa.csv
Shape: (1933, 6)
Columns: ['soru', 'cevap', 'veri türü', 'kaynak', 'context', 'score']


,soru,cevap,veri türü,kaynak,context,score
0,Bayrak Kanunu'nda bayrağın hangi özellikleri b...,"Bayrak Kanunu'nda bayrağın rengi, ölçüleri, şe...",hukuk,Türk Bayrağı Tüzüğü,DÖRDÜNCÜ BÖLÜM\r\nBayrağın Konulabileceği ve Ö...,10
1,Savaşta yalan haber yayma suçunun kapsamı nedir?,"Savaşta yalan haber yayma suçunun kapsamı, sav...",hukuk,Türk Ceza Kanunu,DÖRDÜNCÜ KISIM\r\nMillete ve Devlete Karşı Suç...,9


kaggle_test_qa.csv
Shape: (1933, 6)
Columns: ['soru', 'cevap', 'veri türü', 'kaynak', 'context', 'score']


...

,question,answer
0,Konut dokunulmazlığının korunmasında devletin ...,"Devlet, konut dokunulmazlığını korumakla yüküm..."
1,Milletvekilliği nasıl düşer?,İstifa eden milletvekilinin milletvekilliğinin...


test_qa.csv
Shape: (1500, 2)
Columns: ['question', 'answer']


,question,answer
0,Anayasanın 90. Maddesi Nasıl Uygulanır?,Milletlerarası antlaşmaların TBMM tarafından o...
1,Hukukta 'legitimate expectation' nedir?,"Legitimate expectation, bir kişinin belirli bi..."


retrieval_corpus.csv
Shape: (3775, 5)
Columns: ['chunk_id', 'source_context_id', 'source', 'chunk_text', 'chunk_len']


,chunk_id,source_context_id,source,chunk_text,chunk_len
0,chunk_000000,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,Türk Vatanı ve Milletinin ebedi varlığını ve Y...,263
1,chunk_000001,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,Dünya milletleri ailesinin eşit haklara sahip ...,194


In [18]:
print("Kaggle train sources:")
display(kaggle_train_df["kaynak"].value_counts())

print("Kaggle val sources:")
display(kaggle_val_df["kaynak"].value_counts())

print("Kaggle test sources:")
display(kaggle_test_df["kaynak"].value_counts())

print("Retrieval corpus sources:")
display(corpus_df["source"].value_counts())

Kaggle train sources:


,count
kaynak,
Türk Ceza Kanunu,2392
Türk Medeni Kanunu,2159
Ceza Muhakemesi Kanunu,1342
Türk Borçlar Kanunu,1144
Türkiye Cumhuriyeti Anayasası,1023
Türkiye Cumhuriyeti İş Kanunu,536
Türk Bayrağı Tüzüğü,254
Bilgi Edinme Kanunu,169


Kaggle val sources:


,count
kaynak,
Türk Ceza Kanunu,501
Türk Medeni Kanunu,455
Ceza Muhakemesi Kanunu,297
Türk Borçlar Kanunu,265
Türkiye Cumhuriyeti Anayasası,208
Türkiye Cumhuriyeti İş Kanunu,117
Türk Bayrağı Tüzüğü,55
Bilgi Edinme Kanunu,35


Kaggle test sources:


,count
kaynak,
Türk Ceza Kanunu,488
Türk Medeni Kanunu,465
Ceza Muhakemesi Kanunu,298
Türk Borçlar Kanunu,248
Türkiye Cumhuriyeti Anayasası,226
Türkiye Cumhuriyeti İş Kanunu,120
Türk Bayrağı Tüzüğü,46
Bilgi Edinme Kanunu,42


Retrieval corpus sources:


,count
source,
Türk Medeni Kanunu,1109
Ceza Muhakemesi Kanunu,765
Türk Borçlar Kanunu,639
Türkiye Cumhuriyeti Anayasası,561
Türk Ceza Kanunu,389
Türkiye Cumhuriyeti İş Kanunu,186
Türk Bayrağı Tüzüğü,66
Bilgi Edinme Kanunu,60


In [19]:
qa_sources = set(
    pd.concat([
        kaggle_train_df["kaynak"],
        kaggle_val_df["kaynak"],
        kaggle_test_df["kaynak"]
    ]).dropna().astype(str).unique()
)

corpus_sources = set(
    corpus_df["source"].dropna().astype(str).unique()
)

missing_in_corpus = sorted(list(qa_sources - corpus_sources))
extra_in_corpus = sorted(list(corpus_sources - qa_sources))

print("QA source count:", len(qa_sources))
print("Corpus source count:", len(corpus_sources))

print("\nSources in QA but missing in retrieval corpus:")
for source in missing_in_corpus:
    print("-", source)

print("\nSources in retrieval corpus but not in QA:")
for source in extra_in_corpus:
    print("-", source)

QA source count: 8
Corpus source count: 8

Sources in QA but missing in retrieval corpus:

Sources in retrieval corpus but not in QA:


In [20]:
kvkk_patterns = [
    "kvk",
    "kvkk",
    "kişisel veri",
    "kişisel veriler",
    "kisisel veri",
    "kisisel veriler",
    "ilgili kişi",
    "ilgili kisi",
    "verisi işlenen",
    "verisi islenen",
    "veri sorumlusu",
    "veri kayıt sistemi",
    "veri kayit sistemi",
    "açık rıza",
    "acik riza"
]

def contains_any_pattern(text, patterns):
    text = str(text).lower()
    return any(pattern in text for pattern in patterns)

In [21]:
qa_files = {
    "train_qa": train_df,
    "val_qa": val_df,
    "test_qa": test_df
}

kvkk_like_counts = {}

for name, df in qa_files.items():
    combined_text = (
        df["question"].astype(str).str.lower()
        + " "
        + df["answer"].astype(str).str.lower()
    )

    mask = combined_text.apply(lambda x: contains_any_pattern(x, kvkk_patterns))
    kvkk_like_counts[name] = int(mask.sum())

    print("=" * 100)
    print(name)
    print("Total rows:", len(df))
    print("KVKK-like rows:", mask.sum())
    display(df[mask].head(10))

train_qa
Total rows: 11350
KVKK-like rows: 624


,question,answer
0,Kvk kurulu üyeleri hangi nedenlerle görevden a...,Seçilmek için gereken şartları taşımadıklarını...
58,Kişisel verileri koruma kuruluna şikayet usulü...,"Veri sorumlusuna başvurunun reddedilmesi, veri..."
60,Miras bırakanın vasiyetnamesi nasıl iptal edilir?,"Vasiyetname, hukuka aykırı bir şekilde düzenle..."
63,"Anayasa madde 20, kişisel verilerin korunması ...","Anayasa madde 20, kişisel verilerin korunması ..."
78,"Anayasa madde 135, kamu kurumu niteliğindeki m...","Anayasa madde 135'e göre, kamu kurumu niteliği..."
93,Kişisel veri korumasının hedefi nedir?,Gerçek kişilerin temel hak ve özgürlüklerini k...
114,Kvk kurulu toplantılarında yapılan görüşmeler ...,"Aksi kararlaştırılmadıkça, kvk kurulu toplantı..."
132,Özel mülkümdeki ağaçlar izinsiz kesildi ne yap...,Anayasa'nın 35. maddesi mülkiyet hakkını güven...
135,"Sanık, bir spor salonunda üyelik bilgilerini i...","Sanık, spor salonunda üyelik bilgilerini izins..."
160,Kişisel veriler hangi şartlarda açık rıza olma...,Kvk kanununun 5. maddesinin ikinci fıkrası ve ...


val_qa
Total rows: 2004
KVKK-like rows: 116


,question,answer
21,Kişinin açık rızası olmaksızın kişisel veriler...,"Kanunlarda açıkça öngörülmesi, fiili imkansızl..."
27,"Anayasa madde 138, yargı bağımsızlığının ihlal...","Anayasa madde 138'e göre, yargı bağımsızlığını..."
29,Kvk kurumunun araştırma ve inceleme yapma göre...,Uygulamaları ve mevzuattaki gelişmeleri takip ...
34,İlgili kişinin temel hak ve özgürlüklerine zar...,"Kişisel verilerin işlenme amaçlarının belirli,..."
49,"Anayasa madde 153, anayasa mahkemesi kararları...","Anayasa madde 153'e göre, anayasa mahkemesi ka..."
54,"Anayasa madde 159, hsk'nın kuruluşu ve görevle...","Anayasa madde 159'a göre, hsk'nın kuruluşu ve ..."
60,Kişisel verilerin yurt dışına aktarılması için...,Türkiye’nin veya ilgili kişinin menfaatinin ci...
89,Kişisel verilerin işlenmesi hangi işlemleri iç...,"Elde edilmesi, kaydedilmesi, depolanması, muha..."
105,Kvk kurulu toplantıları için gereken en az üye...,"Kvk kurulu, başkan dahil en az altı üye ile to..."
111,Kvk kanunu'na göre kişisel veriler hangi halle...,Kvk kanununun 5. maddesinin ikinci fıkrası ve ...


test_qa
Total rows: 1500
KVKK-like rows: 86


,question,answer
35,Hangi tür bilgiler özel nitelikli kişisel veri...,"Irkı, etnik kökeni, siyasi düşüncesi, felsefi ..."
37,Kvk kurulu üyeleri hangi durumlarda görevden a...,Seçilmek için gereken şartları taşımadıklarını...
49,Herhangi bir gerçek veya tüzel kişi aynı zaman...,"Veri sorumlusu, kişisel verilerin işleme amaçl..."
55,Kişinin rızası olmadan kişisel veriler hangi ş...,"Kanunlarda açıkça öngörülmesi, fiili imkansızl..."
117,Veri kayıt sistemi nedir?,Kişisel verilerin belirli kriterlere göre yapı...
120,Kişisel verilerin korunması kanunu kapsamındak...,"Kişisel verilerin işlenmesi, korunması ve ilgi..."
131,Kişisel verilerin işlenmesi nasıl yapılır?,Otomatik veya otomatik olmayan yollarla elde e...
184,Kişisel verilerin saklama süreleri nasıl belir...,Veri güvenliğinin sağlanması konusunda veri so...
198,Kişisel verilerin korunması kanunu'nun içerdiğ...,"Kişisel verilerin işlenmesi, korunması ve ilgi..."
214,"Sanık, bir özel şirkette izinsiz olarak çalışa...",Sanığın bir özel şirkette izinsiz olarak çalış...


In [22]:
corpus_combined_text = (
    corpus_df["source"].astype(str).str.lower()
    + " "
    + corpus_df["chunk_text"].astype(str).str.lower()
)

corpus_kvkk_mask = corpus_combined_text.apply(
    lambda x: contains_any_pattern(x, kvkk_patterns)
)

print("Retrieval corpus rows:", len(corpus_df))
print("KVKK-like corpus rows:", corpus_kvkk_mask.sum())

display(corpus_df[corpus_kvkk_mask].head(30))

Retrieval corpus rows: 3775
KVKK-like corpus rows: 15


,chunk_id,source_context_id,source,chunk_text,chunk_len
388,chunk_000388,kaggle_ctx_00018,Türkiye Cumhuriyeti Anayasası,"(Ek fıkra: 7/5/2010-5982/2 md.) Herkes, kendis...",524
439,chunk_000439,kaggle_ctx_00019,Türkiye Cumhuriyeti Anayasası,"(Ek fıkra: 3/10/2001-4709/16 md.) Devlet, işle...",155
604,chunk_000604,kaggle_ctx_00028,Bilgi Edinme Kanunu,"Kamu yararının gerektirdiği hâllerde, kişisel ...",222
647,chunk_000647,kaggle_ctx_00031,Ceza Muhakemesi Kanunu,"(1) 75, 76 ve 78 inci madde hükümlerine göre a...",251
1025,chunk_001025,kaggle_ctx_00050,Ceza Muhakemesi Kanunu,(2) Sanığa veya mağdura ait kişisel verilerin ...,185
1452,chunk_001452,kaggle_ctx_00067,Türk Medeni Kanunu,"Madde 194- Eşlerden biri, diğer eşin açık rıza...",723
1746,chunk_001746,kaggle_ctx_00082,Türk Medeni Kanunu,"İlgili veya yakınları, bu karara karşı bildiri...",423
1747,chunk_001747,kaggle_ctx_00082,Türk Medeni Kanunu,"Madde 437- Hâkim, basit yargılama usulüne göre...",254
2814,chunk_002814,kaggle_ctx_00132,Türk Borçlar Kanunu,MADDE 349- Aile konutu olarak kullanılmak üzer...,631
2925,chunk_002925,kaggle_ctx_00136,Türk Borçlar Kanunu,MADDE 418- İşçi işverenle birlikte ev düzeni i...,631


In [23]:
raw_files = glob.glob(f"{raw_path}/*.csv")

print("Raw CSV files:")
for file in raw_files:
    print(file)

Raw CSV files:
/content/drive/MyDrive/turkish_legal_rag/data/raw/turkish_law_dataset.csv


In [24]:
raw_dataset_path = f"{raw_path}/turkish_law_dataset.csv"

raw_df = pd.read_csv(raw_dataset_path)

print("Raw dataset:", raw_df.shape)
print("Columns:", raw_df.columns.tolist())

display(raw_df.head())

Raw dataset: (13954, 6)
Columns: ['soru', 'cevap', 'veri türü', 'kaynak', 'context', 'Score']


,soru,cevap,veri türü,kaynak,context,Score
0,"Anayasa, Türk Vatanı ve Milletinin ebedi varlı...","Anayasa, Türk Vatanı ve Milletinin ebedi varlı...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,8
1,"Anayasa, Türkiye Cumhuriyetinin hangi milliyet...","Anayasa, Türkiye Cumhuriyetinin kurucusu olan ...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,8
2,"Anayasa, Türkiye Cumhuriyetini hangi konumda t...","Anayasa, Türkiye Cumhuriyetini dünya milletler...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,9
3,"Anayasa, Türkiye Cumhuriyetinin hangi hedefler...","Anayasa, Türkiye Cumhuriyetinin ebedi varlığın...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,9
4,"Anayasa, egemenliğin kime ait olduğunu nasıl b...","Anayasa, egemenliğin kayıtsız şartsız Türk Mil...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,10


In [25]:
raw_combined_text = raw_df.astype(str).agg(" ".join, axis=1).str.lower()

raw_kvkk_mask = raw_combined_text.apply(
    lambda x: contains_any_pattern(x, kvkk_patterns)
)

print("Raw KVKK-like rows:", raw_kvkk_mask.sum())

display(raw_df[raw_kvkk_mask].head(20))

Raw KVKK-like rows: 714


,soru,cevap,veri türü,kaynak,context,Score
110,TBMM'nin araştırma komisyonu nedir?,"TBMM'nin araştırma komisyonu, belli bir konuda...",hukuk,Türkiye Cumhuriyeti Anayasası,ÜÇÜNCÜ KISIM\n\nCUMHURİYETİN TEMEL ORGANLARI\n...,7
171,Bir yasama organının araştırma komisyonu nedir?,"Bir yasama organının araştırma komisyonu, bell...",hukuk,Türkiye Cumhuriyeti Anayasası,ÜÇÜNCÜ KISIM\n\nCUMHURİYETİN TEMEL ORGANLARI\n...,7
264,"Ulusal meclis, üyelerinin üçte ikisinin gizli ...","Türkiye Büyük Millet Meclisi (TBMM), üye tamsa...",hukuk,Türkiye Cumhuriyeti Anayasası,ÜÇÜNCÜ KISIM\n\nCUMHURİYETİN TEMEL ORGANLARI\n...,9
360,Eğitim ve araştırma faaliyetlerini düzenleyen ...,Eğitim ve araştırma faaliyetlerini düzenleyen ...,hukuk,Türkiye Cumhuriyeti Anayasası,ÜÇÜNCÜ KISIM\n\nCUMHURİYETİN TEMEL ORGANLARI\n...,7
363,Bilgiye erişim ve medya özgürlüğünü düzenleyen...,Bilgiye erişim ve medya özgürlüğünü düzenleyen...,hukuk,Türkiye Cumhuriyeti Anayasası,ÜÇÜNCÜ KISIM\n\nCUMHURİYETİN TEMEL ORGANLARI\n...,7
1032,"Madde 17, hangi temel hakkı tanımlar?","Madde 17, kişinin dokunulmazlığı, maddi ve man...",hukuk,Türkiye Cumhuriyeti Anayasası,İKİNCİ KISIM\n\nTEMEL HAKLAR VE ÖDEVLER\nİKİNC...,9
1033,Kişinin vücut bütünlüğüne ne zaman dokunulabilir?,Kişinin vücut bütünlüğüne tıbbi zorunluluklar ...,hukuk,Türkiye Cumhuriyeti Anayasası,İKİNCİ KISIM\n\nTEMEL HAKLAR VE ÖDEVLER\nİKİNC...,8
1034,İşkence ve eziyetin yasaklanması hangi temel h...,"İşkence ve eziyetin yasaklanması, kişinin doku...",hukuk,Türkiye Cumhuriyeti Anayasası,İKİNCİ KISIM\n\nTEMEL HAKLAR VE ÖDEVLER\nİKİNC...,8
1035,Madde 17'deki 'meşru müdafaa' ne anlama gelir?,"Meşru müdafaa, kişinin kendi hayatını veya baş...",hukuk,Türkiye Cumhuriyeti Anayasası,İKİNCİ KISIM\n\nTEMEL HAKLAR VE ÖDEVLER\nİKİNC...,8
1036,Angarya yasaklamasının amacı nedir?,"Angarya yasaklamasının amacı, bireyleri zorla ...",hukuk,Türkiye Cumhuriyeti Anayasası,İKİNCİ KISIM\n\nTEMEL HAKLAR VE ÖDEVLER\nİKİNC...,8


In [26]:
print("Raw kaynak counts:")
display(raw_df["kaynak"].value_counts())

strict_kvkk_source_mask = raw_df["kaynak"].astype(str).str.lower().apply(
    lambda x: (
        "kişisel verilerin korunması" in x
        or "kisisel verilerin korunmasi" in x
        or "kvk" in x
        or "kvkk" in x
    )
)

print("Raw strict KVKK source rows:", strict_kvkk_source_mask.sum())

display(raw_df[strict_kvkk_source_mask].head(20))

Raw kaynak counts:


,count
kaynak,
Türk Ceza Kanunu,3738
Türk Medeni Kanunu,3399
Ceza Muhakemesi Kanunu,2074
Türk Borçlar Kanunu,1791
Türkiye Cumhuriyeti Anayasası,1488
Türkiye Cumhuriyeti İş Kanunu,822
Türk Bayrağı Tüzüğü,388
Bilgi Edinme Kanunu,254


Raw strict KVKK source rows: 0


,soru,cevap,veri türü,kaynak,context,Score


In [27]:
print("Sources of raw KVKK-like rows:")
display(raw_df[raw_kvkk_mask]["kaynak"].value_counts())

Sources of raw KVKK-like rows:


,count
kaynak,
Türk Medeni Kanunu,181
Ceza Muhakemesi Kanunu,163
Türk Borçlar Kanunu,127
Türk Ceza Kanunu,104
Türkiye Cumhuriyeti Anayasası,98
Bilgi Edinme Kanunu,39
Türk Bayrağı Tüzüğü,1
Türkiye Cumhuriyeti İş Kanunu,1


In [28]:
def normalize_question(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zçğıöşü0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

kaggle_all_df = pd.concat(
    [kaggle_train_df, kaggle_val_df, kaggle_test_df],
    ignore_index=True
)

final_all_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

kaggle_questions = set(kaggle_all_df["soru"].apply(normalize_question))

final_all_df["normalized_question"] = final_all_df["question"].apply(normalize_question)
final_all_df["exists_in_kaggle"] = final_all_df["normalized_question"].isin(kaggle_questions)

print("Final QA total:", len(final_all_df))
print("Exists in Kaggle:", final_all_df["exists_in_kaggle"].sum())
print("Not in Kaggle:", (~final_all_df["exists_in_kaggle"]).sum())
print("Exist ratio:", final_all_df["exists_in_kaggle"].mean())

Final QA total: 14854
Exists in Kaggle: 38
Not in Kaggle: 14816
Exist ratio: 0.002558233472465329


In [29]:
final_all_df["is_kvkk_like"] = (
    final_all_df["question"].astype(str).str.lower()
    + " "
    + final_all_df["answer"].astype(str).str.lower()
).apply(lambda x: contains_any_pattern(x, kvkk_patterns))

kvkk_final_df = final_all_df[final_all_df["is_kvkk_like"]].copy()

print("Final KVKK-like QA rows:", len(kvkk_final_df))
print("KVKK-like exists in Kaggle:", kvkk_final_df["exists_in_kaggle"].sum())
print("KVKK-like not in Kaggle:", (~kvkk_final_df["exists_in_kaggle"]).sum())

display(kvkk_final_df[["question", "answer", "exists_in_kaggle"]].head(20))

Final KVKK-like QA rows: 826
KVKK-like exists in Kaggle: 0
KVKK-like not in Kaggle: 826


,question,answer,exists_in_kaggle
0,Kvk kurulu üyeleri hangi nedenlerle görevden a...,Seçilmek için gereken şartları taşımadıklarını...,False
58,Kişisel verileri koruma kuruluna şikayet usulü...,"Veri sorumlusuna başvurunun reddedilmesi, veri...",False
60,Miras bırakanın vasiyetnamesi nasıl iptal edilir?,"Vasiyetname, hukuka aykırı bir şekilde düzenle...",False
63,"Anayasa madde 20, kişisel verilerin korunması ...","Anayasa madde 20, kişisel verilerin korunması ...",False
78,"Anayasa madde 135, kamu kurumu niteliğindeki m...","Anayasa madde 135'e göre, kamu kurumu niteliği...",False
93,Kişisel veri korumasının hedefi nedir?,Gerçek kişilerin temel hak ve özgürlüklerini k...,False
114,Kvk kurulu toplantılarında yapılan görüşmeler ...,"Aksi kararlaştırılmadıkça, kvk kurulu toplantı...",False
132,Özel mülkümdeki ağaçlar izinsiz kesildi ne yap...,Anayasa'nın 35. maddesi mülkiyet hakkını güven...,False
135,"Sanık, bir spor salonunda üyelik bilgilerini i...","Sanık, spor salonunda üyelik bilgilerini izins...",False
160,Kişisel veriler hangi şartlarda açık rıza olma...,Kvk kanununun 5. maddesinin ikinci fıkrası ve ...,False


In [30]:
eval20_df = test_df.sample(n=20, random_state=42).reset_index(drop=True)

eval20_df["combined_text"] = (
    eval20_df["question"].astype(str).str.lower()
    + " "
    + eval20_df["answer"].astype(str).str.lower()
)

eval20_df["is_kvkk_like"] = eval20_df["combined_text"].apply(
    lambda x: contains_any_pattern(x, kvkk_patterns)
)

eval20_df["is_gecici_madde_20"] = eval20_df["combined_text"].str.contains(
    "geçici madde 20|gecici madde 20|20 mayıs 2016|20/5/2016|20.05.2016",
    regex=True,
    na=False
)

eval20_df["coverage_issue"] = (
    eval20_df["is_kvkk_like"]
    | eval20_df["is_gecici_madde_20"]
)

display(eval20_df[[
    "question",
    "answer",
    "is_kvkk_like",
    "is_gecici_madde_20",
    "coverage_issue"
]])

,question,answer,is_kvkk_like,is_gecici_madde_20,coverage_issue
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...,False,False,False
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ...",False,False,False
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ...",False,False,False
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,False,True,True
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...,False,False,False
5,Cumhurbaşkanının yemin etme zorunluluğu nedir,"Cumhurbaşkanının yemin etme zorunluluğu, anaya...",False,False,False
6,Anayasanın 122. Maddesi Nedir?,"Anayasanın 122. maddesi, sıkıyönetim ilanı ve ...",False,False,False
7,Yasama dokunulmazlığının kaldırılmasına karşı ...,Yasama dokunulmazlığının kaldırılmasına veya m...,False,False,False
8,"Bir siyasi parti, çalışma şartlarının belirli ...","Evet, Anayasanın 50. Maddesi, herkesin çalışma...",False,False,False
9,"Devlet Denetleme Kurulu, Silahlı Kuvvetler üze...","Anayasanın 108. Maddesi, Silahlı Kuvvetlerin D...",False,False,False


In [31]:
print("Eval20 total:", len(eval20_df))
print("KVKK-like rows in eval20:", eval20_df["is_kvkk_like"].sum())
print("Geçici Madde 20 rows in eval20:", eval20_df["is_gecici_madde_20"].sum())
print("Coverage issue rows in eval20:", eval20_df["coverage_issue"].sum())

display(eval20_df[eval20_df["coverage_issue"]])

Eval20 total: 20
KVKK-like rows in eval20: 2
Geçici Madde 20 rows in eval20: 1
Coverage issue rows in eval20: 3


,question,answer,combined_text,is_kvkk_like,is_gecici_madde_20,coverage_issue
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,geçici madde 20 ne zaman eklendi? 20 mayıs 201...,False,True,True
11,İlgili kişi ne demektir?,Kişisel verisi işlenen gerçek kişi demektir.,i̇lgili kişi ne demektir? kişisel verisi işlen...,True,False,True
15,Kişisel veriler hangi mevzuata göre işlenir?,Kvk kanununda ve diğer kanunlarda öngörülen me...,kişisel veriler hangi mevzuata göre işlenir? k...,True,False,True


In [32]:
coverage_terms = [
    "Geçici Madde 20",
    "geçici madde 20",
    "Geçici 20",
    "20 Mayıs 2016",
    "20/5/2016",
    "20.05.2016",
    "20 mayıs 2016"
]

for term in coverage_terms:
    mask = corpus_df["chunk_text"].astype(str).str.lower().str.contains(term.lower(), na=False)
    print(term, "matches:", mask.sum())

    if mask.sum() > 0:
        display(corpus_df[mask].head(10))

Geçici Madde 20 matches: 0
geçici madde 20 matches: 0
Geçici 20 matches: 1


,chunk_id,source_context_id,source,chunk_text,chunk_len
3745,chunk_003745,kaggle_ctx_00237,Türkiye Cumhuriyeti İş Kanunu,31/7/2008-5797/10 md.) Bu fıkrada düzenlenen t...,796


20 Mayıs 2016 matches: 0
20/5/2016 matches: 0
20.05.2016 matches: 0
20 mayıs 2016 matches: 0


In [33]:
source_aware_results_path = f"{metrics_path}/source_aware_article_aware_turkish_bge_rag_testset_scored.csv"

source_aware_results_df = pd.read_csv(source_aware_results_path)

print(source_aware_results_df.shape)
display(source_aware_results_df.head())

(20, 16)


,question,expected_answer,generated_answer,clean_generated_answer,top1_chunk_id,top1_source,top1_source_filter,top1_context,top1_article_bonus,top1_original_rank,top1_rerank_rank,top1_rerank_score,top1_fusion_score,retrieved_contexts,is_valid_sample,manual_score
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...,"Anayasanın 101. Maddesiyle ilgili tartışmalar,...","Anayasanın 101. Maddesiyle ilgili tartışmalar,...",chunk_000537,Türkiye Cumhuriyeti Anayasası,Türkiye Cumhuriyeti Anayasası,Cumhurbaşkanlığı seçiminde birinci oylamada ge...,1.0,1,1,0.023523,1.00,Cumhurbaşkanlığı seçiminde birinci oylamada ge...,True,0.0
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ...","Anayasanın 10. Maddesi, herkesin dil, ırk, ren...","Anayasanın 10. Maddesi, herkesin dil, ırk, ren...",chunk_000274,Türkiye Cumhuriyeti Anayasası,Türkiye Cumhuriyeti Anayasası,"Madde 10 – Herkes, dil, ırk, renk, cinsiyet, s...",1.0,1,1,0.027961,1.00,"Madde 10 – Herkes, dil, ırk, renk, cinsiyet, s...",True,0.5
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ...","Hayır. Anayasanın 17. Maddesi, herkesin yaşama...","Hayır. Anayasanın 17. Maddesi, herkesin yaşama...",chunk_000373,Türkiye Cumhuriyeti Anayasası,Türkiye Cumhuriyeti Anayasası,"Madde 17 – Herkes, yaşama, maddi ve manevi var...",1.0,1,1,0.385170,1.00,"Madde 17 – Herkes, yaşama, maddi ve manevi var...",True,0.5
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,Geçici madde 20 13/5/1981 gün eklendi.,Geçici madde 20 13/5/1981 gün eklendi.,chunk_003745,Türkiye Cumhuriyeti İş Kanunu,NaN,31/7/2008-5797/10 md.) Bu fıkrada düzenlenen t...,1.0,1,1,0.438255,1.00,31/7/2008-5797/10 md.) Bu fıkrada düzenlenen t...,True,0.0
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...,"Evet, TCK 121 madde hakkının kullanılmasının e...","Evet, TCK 121 madde hakkının kullanılmasının e...",chunk_003431,Türk Ceza Kanunu,Türk Ceza Kanunu,ÜÇÜNCÜ KISIM\r\nTopluma Karşı Suçlar ALTINCI B...,0.0,1,5,0.002067,0.76,ÜÇÜNCÜ KISIM\r\nTopluma Karşı Suçlar ALTINCI B...,True,0.5


In [34]:
source_aware_results_df["coverage_issue"] = eval20_df["coverage_issue"].values
source_aware_results_df["is_kvkk_like"] = eval20_df["is_kvkk_like"].values
source_aware_results_df["is_gecici_madde_20"] = eval20_df["is_gecici_madde_20"].values

display(source_aware_results_df[[
    "question",
    "manual_score",
    "is_valid_sample",
    "coverage_issue",
    "is_kvkk_like",
    "is_gecici_madde_20"
]])

,question,manual_score,is_valid_sample,coverage_issue,is_kvkk_like,is_gecici_madde_20
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,0.0,True,False,False,False
1,"Bir grup vatandaş, belirli bir etnik grubun di...",0.5,True,False,False,False
2,"Bir grup akademisyen, yaşama hakkının sınırlan...",0.5,True,False,False,False
3,Geçici madde 20 ne zaman eklendi?,0.0,True,True,False,True
4,Videoda TCK 121 ihlali sabit değil mi?,0.5,True,False,False,False
5,Cumhurbaşkanının yemin etme zorunluluğu nedir,0.0,True,False,False,False
6,Anayasanın 122. Maddesi Nedir?,0.0,True,False,False,False
7,Yasama dokunulmazlığının kaldırılmasına karşı ...,1.0,True,False,False,False
8,"Bir siyasi parti, çalışma şartlarının belirli ...",0.5,True,False,False,False
9,"Devlet Denetleme Kurulu, Silahlı Kuvvetler üze...",0.0,True,False,False,False


In [35]:
valid_all_df = source_aware_results_df[
    source_aware_results_df["is_valid_sample"] == True
].copy()

valid_coverage_clean_df = valid_all_df[
    valid_all_df["coverage_issue"] == False
].copy()

print("Original valid count:", len(valid_all_df))
print("Original score:", valid_all_df["manual_score"].mean())

print("Coverage-clean valid count:", len(valid_coverage_clean_df))
print("Coverage-clean score:", valid_coverage_clean_df["manual_score"].mean())

Original valid count: 19
Original score: 0.42105263157894735
Coverage-clean valid count: 16
Coverage-clean score: 0.46875


In [36]:
coverage_summary = {
    "kaggle_qa_rows": len(kaggle_all_df),
    "final_qa_rows": len(final_all_df),

    "final_qa_exists_in_kaggle": int(final_all_df["exists_in_kaggle"].sum()),
    "final_qa_not_in_kaggle": int((~final_all_df["exists_in_kaggle"]).sum()),
    "final_qa_exists_in_kaggle_ratio": float(final_all_df["exists_in_kaggle"].mean()),

    "final_qa_kvkk_like_rows": int(kvkk_final_df.shape[0]),
    "final_qa_kvkk_like_exists_in_kaggle": int(kvkk_final_df["exists_in_kaggle"].sum()),
    "final_qa_kvkk_like_not_in_kaggle": int((~kvkk_final_df["exists_in_kaggle"]).sum()),

    "retrieval_corpus_rows": len(corpus_df),
    "retrieval_corpus_source_count": int(corpus_df["source"].nunique()),
    "retrieval_corpus_kvkk_like_rows": int(corpus_kvkk_mask.sum()),

    "raw_rows": len(raw_df),
    "raw_kvkk_like_rows": int(raw_kvkk_mask.sum()),
    "raw_strict_kvkk_source_rows": int(strict_kvkk_source_mask.sum()),

    "eval20_total_rows": len(eval20_df),
    "eval20_kvkk_like_rows": int(eval20_df["is_kvkk_like"].sum()),
    "eval20_gecici_madde_20_rows": int(eval20_df["is_gecici_madde_20"].sum()),
    "eval20_coverage_issue_rows": int(eval20_df["coverage_issue"].sum()),

    "source_aware_original_score": float(valid_all_df["manual_score"].mean()),
    "source_aware_original_valid_count": int(len(valid_all_df)),

    "source_aware_coverage_clean_score": float(valid_coverage_clean_df["manual_score"].mean()),
    "source_aware_coverage_clean_valid_count": int(len(valid_coverage_clean_df)),
}

coverage_summary_df = pd.DataFrame([coverage_summary])
coverage_summary_df

,kaggle_qa_rows,final_qa_rows,final_qa_exists_in_kaggle,final_qa_not_in_kaggle,final_qa_exists_in_kaggle_ratio,final_qa_kvkk_like_rows,final_qa_kvkk_like_exists_in_kaggle,final_qa_kvkk_like_not_in_kaggle,retrieval_corpus_rows,retrieval_corpus_source_count,...,raw_kvkk_like_rows,raw_strict_kvkk_source_rows,eval20_total_rows,eval20_kvkk_like_rows,eval20_gecici_madde_20_rows,eval20_coverage_issue_rows,source_aware_original_score,source_aware_original_valid_count,source_aware_coverage_clean_score,source_aware_coverage_clean_valid_count
0,12885,14854,38,14816,0.002558,826,0,826,3775,8,...,714,0,20,2,1,3,0.421053,19,0.46875,16


In [37]:
coverage_summary_df.to_csv(
    f"{metrics_path}/data_coverage_analysis_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

source_aware_results_df.to_csv(
    f"{metrics_path}/source_aware_results_with_coverage_flags.csv",
    index=False,
    encoding="utf-8-sig"
)

eval20_df.to_csv(
    f"{metrics_path}/eval20_coverage_flags.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame({
    "missing_in_corpus": missing_in_corpus
}).to_csv(
    f"{metrics_path}/sources_missing_in_retrieval_corpus.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Coverage analysis files saved.")

Coverage analysis files saved.


## Coverage Analysis Conclusion

The analysis showed a clear mismatch between the final QA files and the current retrieval corpus.

The final QA files contain many KVKK / personal data related questions. However, the current retrieval corpus does not contain a dedicated KVKK legal source. The raw dataset also does not include a strict KVKK source.

Only 38 out of 14,854 final QA questions were matched with the Kaggle QA questions, which indicates that the final QA split and the retrieval corpus are not fully aligned.

In the 20-sample evaluation set, 3 samples were marked as coverage issues: two KVKK-related samples and one Geçici Madde 20 sample. The original source-aware RAG score was 0.421053, while the coverage-clean score increased to 0.46875.

This suggests that part of the remaining error is caused by corpus coverage mismatch, not only by retrieval or generation quality.